# Chapter 14: Fine-tuning & Adaptation Techniques
## Notebook 01 — Fine-tuning Basics

This notebook is the bridge from prompting and RAG (Chapters 11–13) into fine-tuning. We cover **when to fine-tune**, **how to format an instruction dataset**, the mechanics of **supervised fine-tuning (SFT)**, and how to **evaluate** the result honestly.

### What you'll learn

| Topic | Section |
|-------|--------|
| Decision tree: prompt vs RAG vs fine-tune | §2 |
| Instruction dataset preparation | §3 |
| SFT concepts: loss masking, padding, packing | §4 |
| Tiny SFT loop (sklearn / numpy analog) | §5 |
| Sketch of Hugging Face `Trainer` workflow | §6 |
| Evaluation basics: held-out, EM, F1 | §7 |

**Estimated time:** 2 hours

---
*Generated by Berta AI*

---
## 1. Setup

We use NumPy, pandas, and scikit-learn. Heavy frameworks (`transformers`, `peft`, `trl`) are optional and wrapped in `try/except`.

In [ ]:
import os
import sys
import json
from pathlib import Path

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 4)
np.random.seed(42)

import config
from dataset_utils import (
    format_instruction, load_jsonl, train_val_split,
    tokenize_budget, pack_examples, build_loss_mask, InstructionDataset,
)
from training_utils import simple_sft_loop, EvalHarness, exact_match, token_f1

print('Setup complete. MAX_SEQ_LEN =', config.MAX_SEQ_LEN)

---
## 2. When to Fine-tune (vs Prompt vs RAG)

Before you fine-tune, ask: *can prompting or retrieval solve this?* Fine-tuning is the right tool when:

- Your task has a **fixed, narrow output format** (style, JSON shape, terse answers) that prompts struggle to enforce.
- You have **plenty of labeled examples** (>= a few hundred high-quality pairs).
- You need **lower latency or smaller models** at inference.
- The behavior is **stable** — the world doesn't change under your feet (otherwise RAG keeps you fresher).

Use the decision table below as a quick guide.

In [ ]:
decision = pd.DataFrame([
    {'need': 'Up-to-date facts',          'prompt': 'no',  'rag': 'yes', 'fine_tune': 'no'},
    {'need': 'Custom style / format',     'prompt': 'ok',  'rag': 'no',  'fine_tune': 'yes'},
    {'need': 'Lower inference cost',      'prompt': 'no',  'rag': 'no',  'fine_tune': 'yes'},
    {'need': 'Few-shot, < 50 examples',   'prompt': 'yes', 'rag': 'maybe', 'fine_tune': 'no'},
    {'need': 'Domain knowledge cutover',  'prompt': 'no',  'rag': 'yes', 'fine_tune': 'maybe'},
    {'need': 'Strict structured output',  'prompt': 'ok',  'rag': 'no',  'fine_tune': 'yes'},
])
decision

---
## 3. Instruction Dataset Preparation

An **instruction dataset** has three fields per row: `instruction`, `input` (optional), and `output`. We format each row into a single text string with a clear separator that tells the model *where the response begins*.

Always do a **deterministic** train/val split so eval numbers are comparable across runs.

In [ ]:
data_path = Path('..') / 'datasets' / 'instructions.jsonl'
rows = load_jsonl(data_path)
print(f'Loaded {len(rows)} examples')

ex = format_instruction(**{k: rows[0][k] for k in ('instruction', 'input', 'output')})
print('--- Prompt portion (model conditions on this) ---')
print(ex['prompt'])
print('--- Response portion (loss is computed only here) ---')
print(ex['response'])

In [ ]:
train, val = train_val_split(rows, train_fraction=0.8, seed=config.RANDOM_SEED)
print(f'train={len(train)}  val={len(val)}')

ds = InstructionDataset(rows=rows)
formatted = ds.formatted()
summary = tokenize_budget(formatted, max_seq_len=64)
print('Token budget summary (whitespace tokens, max_seq_len=64):', summary)

### Packing short examples

Many instruction examples are short. **Packing** concatenates them up to the sequence length so the GPU isn't wasting compute on padding. The trade-off: you must mask cross-example attention boundaries (real frameworks handle this) and ensure you still mask out prompt tokens for the loss.

In [ ]:
packed = pack_examples(formatted, max_seq_len=64)
print(f'{len(formatted)} raw examples -> {len(packed)} packed sequences')
print('First packed sequence (truncated):')
print(packed[0][:300], '...')

---
## 4. SFT Concepts: Loss Masking, Padding, Targets

The single biggest mistake in instruction tuning is **supervising the prompt**. The model already saw the prompt at inference; we want it to learn to *generate the response*. So:

1. Tokenize `prompt + response` together.
2. Build a **loss mask** that is `0` for prompt tokens and `1` for response tokens (and `0` for padding).
3. Compute cross-entropy and average **only over the masked-in positions**.

Below we visualize what the mask looks like for one example.

In [ ]:
prompt_tokens = ex['prompt'].split()
resp_tokens = ex['response'].split()
mask = build_loss_mask(prompt_len=len(prompt_tokens), total_len=len(prompt_tokens) + len(resp_tokens))
for tok, m in list(zip(prompt_tokens + resp_tokens, mask))[:18]:
    role = 'response' if m else 'prompt  '
    print(f'  {role}  mask={m}  token={tok}')
print(f'... ({len(mask)} positions total, {sum(mask)} supervised)')

---
## 5. A Tiny SFT Loop

Real fine-tuning trains a transformer's last linear head (and earlier layers). Here we use a **linear head on hand-built features** as the analog: same loop shape, but it runs in seconds on a CPU.

We featurize each instruction with TF-IDF, predict a label (the *kind* of instruction), and use a per-token loss mask of all-ones (every example is supervised). The loop has:

- linear warmup + cosine decay learning rate,
- gradient clipping,
- early stopping on validation loss,
- separate train/val histories.

This is exactly the structure your real `Trainer` will use.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

labels = ['translate', 'math', 'summary', 'code', 'qa', 'sentiment']
def assign_label(instr: str) -> int:
    s = instr.lower()
    if 'translate' in s: return 0
    if any(k in s for k in ('sum', 'add', 'multiply', 'product', 'plus')): return 1
    if 'summar' in s: return 2
    if 'python' in s or 'function' in s or 'code' in s: return 3
    if 'sentiment' in s or 'positive' in s or 'negative' in s: return 5
    return 4

texts = [r['instruction'] + ' ' + (r.get('input') or '') for r in rows]
y = np.array([assign_label(r['instruction']) for r in rows])

vec = TfidfVectorizer(min_df=1, ngram_range=(1, 2))
X = vec.fit_transform(texts).toarray().astype(np.float32)
mask = np.ones(len(rows), dtype=int)
print('Feature matrix:', X.shape, '  classes:', sorted(set(y.tolist())))

In [ ]:
out = simple_sft_loop(
    X, y, mask, n_classes=len(labels),
    epochs=20, batch_size=4, lr=0.5,
    warmup_ratio=0.1, weight_decay=1e-3,
    grad_clip=1.0, val_split=0.25,
    early_stop_patience=4, seed=config.RANDOM_SEED,
)
h = out['history']
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(h['train_loss'], label='train'); ax[0].plot(h['val_loss'], label='val')
ax[0].set_title('Loss (overfitting check)'); ax[0].legend(); ax[0].set_xlabel('epoch')
ax[1].plot(h['lr']); ax[1].set_title('Learning rate (warmup + cosine)'); ax[1].set_xlabel('epoch')
plt.tight_layout(); plt.show()
print(f'Final val loss: {h["val_loss"][-1]:.4f}')

**Reading the curves**: train falling while val rises is **overfitting** — fine-tuning reaches it quickly because pre-trained models already know a lot. Cures: reduce epochs, lower LR, add regularization, or drop to a small LoRA rank (Notebook 02).

---
## 6. Sketch: Hugging Face `Trainer` Workflow

On a real model, the loop above is replaced by `transformers.Trainer` (or `trl.SFTTrainer`). It handles tokenization, batching, mixed precision, gradient accumulation, distributed training, and checkpointing.

We wrap the import so the cell stays useful even without `transformers` installed.

In [ ]:
try:
    from transformers import (
        AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer,
        DataCollatorForLanguageModeling,
    )
    print('transformers is installed. Sketch:')
    sketch = '''
tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

def encode(ex):
    full = ex["prompt"] + ex["response"]
    enc = tok(full, truncation=True, max_length=512)
    prompt_len = len(tok(ex["prompt"])["input_ids"])
    labels = [-100] * prompt_len + enc["input_ids"][prompt_len:]
    enc["labels"] = labels[: len(enc["input_ids"])]
    return enc

args = TrainingArguments(
    output_dir="./out", per_device_train_batch_size=4,
    learning_rate=2e-4, num_train_epochs=3,
    warmup_ratio=0.03, lr_scheduler_type="cosine",
    eval_strategy="epoch", save_strategy="epoch", logging_steps=10,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds)
trainer.train()
'''
    print(sketch)
except ImportError:
    print('transformers not installed. Workflow:')
    print('  1) load tokenizer + base model')
    print('  2) tokenize prompt+response, mask prompt tokens with label_id=-100')
    print('  3) Trainer with cosine schedule, warmup, eval_strategy="epoch"')
    print('  4) save model + tokenizer; register version (see Notebook 03).')

---
## 7. Evaluation Basics

Two metrics give you a first read on quality:

- **Exact match** — does the prediction equal the reference (after normalization)?
- **Token-F1** — overlap of tokens, balancing precision and recall.

Both are computed on a **held-out set** that the model never saw during training. Report mean and per-task slices, not just an overall number.

In [ ]:
eval_path = Path('..') / 'datasets' / 'eval_set.jsonl'
eval_rows = load_jsonl(eval_path)
refs = [r['reference'] for r in eval_rows]

# Two toy 'model' predictions to demonstrate the harness.
preds_a = [r['reference'] for r in eval_rows]                         # perfect copy
preds_b = [r['reference'].split()[0] if r['reference'] else '' for r in eval_rows]  # only first token

harness = EvalHarness(references=refs)
print('Model A:', harness.score(preds_a))
print('Model B:', harness.score(preds_b))
print('Win rate A vs B:', harness.compare(preds_a, preds_b))

---
## 8. Key Takeaways

- **Choose carefully**: prompt < RAG < fine-tune in order of effort and binding cost.
- **Format consistently**: a fixed instruction template is half the battle for SFT.
- **Mask the loss**: supervise the response, not the prompt.
- **Schedule the LR**: warmup + cosine decay is the dependable default.
- **Evaluate held-out**: report EM and F1 (and slices), not just train loss.

Next — **Notebook 02**: implement LoRA from scratch and learn why most modern fine-tuning is parameter-efficient.

---
*Generated by Berta AI*